<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Exercises — Five Ways to Call a Model

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Practise

Every exercise does the same job — send text to a model, get something back. Five different front doors.

| # | Library | What you call | Model |
|---|---|---|---|
| **Q1** | `openai` | `client.chat.completions.create()` | `gpt-4o-mini` · `gemini-3.6-flash` |
| **Q2** | `litellm` | `completion()` | `openai/gpt-4o-mini` · `gemini/gemini-3.6-flash` |
| **Q3** | `litellm` | `embedding()` | `text-embedding-3-small` |
| **Q4** | `sentence-transformers` | `SentenceTransformer(...).encode()` | `all-MiniLM-L6-v2` |
| **Q5** | `langchain` | `.invoke()` | `ChatOpenAI` · `ChatGoogleGenerativeAI` |

**Each cell gives you the steps — you write the code.** Installs and imports are done for you at the top; after that, read the hint and write it yourself. Run a cell with **Shift + Enter**.

> **Q4 needs no API key** — it downloads a small model and runs it on the Colab CPU.
> Q1, Q2, Q3 and Q5 need an OpenAI key; the Gemini halves of Q1, Q2 and Q5 need a Gemini key.

---

## 1. Setup

Run these three cells. Nothing to write.

In [ ]:
# PROVIDED - just run this cell. Takes about a minute.
!pip install -q openai litellm sentence-transformers langchain langchain-openai langchain-google-genai

In [ ]:
# PROVIDED - just run this cell. Every import the notebook needs, all five exercises.
import os
import numpy as np
from getpass import getpass

from openai import OpenAI                                     # Q1
from litellm import completion, embedding                     # Q2, Q3
from sentence_transformers import SentenceTransformer         # Q4
from langchain_openai import ChatOpenAI                       # Q5
from langchain_google_genai import ChatGoogleGenerativeAI     # Q5

print("Imports ready")

In [ ]:
# PROVIDED - just run this cell. Q3 and Q4 both use these.
sentences = [
    "The cat sat on the mat.",
    "A kitten rested on the rug.",
    "Chennai is a coastal city in Tamil Nadu.",
]

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(len(sentences), "sentences ready")

### Your turn — the keys

From here on, **you write every cell.**

In [ ]:
# Read your keys in with getpass - never type a key straight into a code cell,
# it gets saved with the notebook.
#
# 1. Ask for the OpenAI key, put it in os.environ as  OPENAI_API_KEY
# 2. Ask for the Gemini key - optional, let a plain Enter skip it.
#    If it was given, set it TWICE:  GEMINI_API_KEY  (LiteLLM looks for this name)
#                               and  GOOGLE_API_KEY  (LangChain looks for this one)
# 3. Define two constants the rest of the notebook will reuse:
#       OPENAI_MODEL = "gpt-4o-mini"
#       GEMINI_MODEL = "gemini-3.6-flash"    # no access yet? use "gemini-2.0-flash"
# 4. Print them, so you can see what actually got set.


# your code here

---

## Q1 — The `openai` client

The official SDK. Build a `client` once, then call `chat.completions.create()` with a list of
`messages`. Each message has a **role** (`system` sets the behaviour, `user` asks the question)
and **content**.

### Q1a — OpenAI (`gpt-4o-mini`)

In [ ]:
# Hint: build the client with  OpenAI()  - no arguments. It picks up OPENAI_API_KEY
#       from the environment on its own.
#       Then call  client.chat.completions.create(model=..., messages=[...])
#       messages is a LIST of dicts, each one with a "role" and a "content".
#       Use a "system" message to set the behaviour, a "user" message to ask.
#       The reply text sits at  resp.choices[0].message.content
#
# Ask it: "Explain what an API is in 2 sentences."


# your code here

### Q1b — Gemini, through the **same** client

Google publishes an **OpenAI-compatible endpoint**. Same class, same method, same response shape.
Exactly three things change: the `api_key`, the `base_url`, and the `model`.

In [ ]:
# Hint: build a SECOND client from the same OpenAI class, but pass two arguments:
#         api_key  = the Gemini key out of os.environ   (which name did you set it under?)
#         base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
#       Pass the key explicitly - leave it out and it quietly uses your OpenAI key.
#       Then call it exactly like Q1a, but with GEMINI_MODEL.
#
# Ask the same question, and compare the two answers.


# your code here

**Two different companies' models, one Python class.** The only thing that sent that call to
Google instead of OpenAI was a URL.

Now look at what it cost you: a *second* client object, hardcoded, with a URL you had to look up.
Q2 removes both.

---

## Q2 — LiteLLM `completion()`

No client object at all — one function, and the **provider lives inside the model string**:
`provider/model`. LiteLLM finds the matching key in your environment on its own.

### Q2a — `openai/gpt-4o-mini`

In [ ]:
# Hint: no client to build. Call  completion(model=..., messages=[...])  directly.
#       The provider lives INSIDE the model string, as  "provider/model".
#       For OpenAI that is  "openai/" + the model name.
#       Pass a temperature of 0.7 while you are there.
#       The response reads exactly like Q1:  .choices[0].message.content
#
# Ask it: "Name one famous scientist and their field."


# your code here

### Q2b — the same call, sent to Gemini

**Change one string.** Nothing else in the cell moves.

In [ ]:
# Hint: copy Q2a and change ONE string. The provider prefix for Google is  gemini
#       so the model becomes  "gemini/" + the Gemini model name.
#       Print  response.usage.total_tokens  too, to see what the call cost.


# your code here

**Compare the diff.** Switching provider in Q1 meant a new client, a new key and a URL.
Here it was one string, in one place — the thing you would put in a config file.

---

## Q3 — LiteLLM `embedding()` → `text-embedding-3-small`

Same library as Q2, **different function**. An embedding does not answer a question — it turns
text into a **list of numbers that carries its meaning**, so you can measure how close two pieces
of text are.

`text-embedding-3-small` is OpenAI's cheap default: **1536 numbers** per piece of text.

In [ ]:
# Hint: same library as Q2, different function:  embedding(model=..., input=...)
#       input takes a LIST of strings - pass the `sentences` list from setup.
#       The model name is  "text-embedding-3-small"  (no provider prefix needed here).
#       Each vector sits at  response.data[i]["embedding"]  - collect them with
#       a list comprehension into  np.array(...)  so you can do maths on them.
#
# Then:
#   1. print the array's .shape        - how many rows, how many numbers each?
#   2. use cosine() to score sentence 0 against sentence 1   (same meaning, no shared words)
#   3. use cosine() to score sentence 0 against sentence 2   (different topic entirely)
#      Which pair scores higher - and did any word have to match for that to work?


# your code here

**Not one word is shared** between *"The cat sat on the mat"* and *"A kitten rested on the rug"* —
and they still score high. Keyword search would have found nothing here.

---

## Q4 — HuggingFace `sentence-transformers` → `all-MiniLM-L6-v2`

The same job with **no API and no key**. The model downloads once (~90 MB) and runs on the Colab
CPU. **384 numbers** per sentence instead of 1536.

In [ ]:
# Hint: build the model with  SentenceTransformer("all-MiniLM-L6-v2")
#       then call  .encode(sentences)  - same list, and it hands back numpy directly,
#       so there is nothing to unpack this time.
#       The first run downloads about 90 MB. Give it a minute.
#
# Then print the .shape and run the SAME two cosine() comparisons you ran in Q3.
# Put the two sets of numbers side by side. Do the SCORES match? Does the RANKING?


# your code here

**Different numbers, same ranking.** Both models agree on *which* sentences belong together, and
disagree on the exact score.

Two things follow, and both matter later:

1. **Never compare a score from one model against a score from another** — compare rankings.
2. **Never mix models on the two sides of one comparison.** Query and stored text must come from
   the same model, or the answer is meaningless.

| | LiteLLM → OpenAI | sentence-transformers |
|---|---|---|
| numbers per sentence | 1536 | 384 |
| cost | billed per token | free |
| runs on | OpenAI's servers | this Colab CPU |
| needs a key | yes | no |
| offline / private data | no | yes |

---

## Q5 — LangChain chat models

LangChain wraps every provider in one object with one method: **`.invoke()`**. That uniformity is
what lets you build chains — and later agents — without the code caring who is behind them.

| Provider | Package (pip) | Class (import) |
|---|---|---|
| OpenAI | `langchain-openai` | `ChatOpenAI` |
| Google Gemini | `langchain-google-genai` | `ChatGoogleGenerativeAI` |

> Worth noticing while you are here: the **pip name uses hyphens**, the **import uses underscores**.

### Q5a — `ChatOpenAI`

In [ ]:
# Hint: build it with  ChatOpenAI(model=..., temperature=0)  - it reads OPENAI_API_KEY
#       from the environment, same as Q1a did.
#       Then call  .invoke("your question")  - a plain string, no messages list.
#       It hands back a message OBJECT, not a string. The text is on  .content
#
# Ask it: "Give me 3 tips to prepare for a coding interview."


# your code here

### Q5b — `ChatGoogleGenerativeAI`

Different class, different package — **identical `.invoke()`**. That is the whole point of the
wrapper.

In [ ]:
# Hint: ChatGoogleGenerativeAI(model=..., temperature=0) - the constructor takes  model=
#       exactly like ChatOpenAI did, and it looks for GOOGLE_API_KEY in the environment.
#       Then  .invoke(...)  and  .content  - identical to Q5a.
#
# Ask it the same question.


# your code here

### Q5c — why anyone bothers

Because both objects answer to the same method, code written against one works against the other
**untouched**. Prove it.

In [ ]:
# Hint: write ONE function  ask(model, question)  that calls  model.invoke(question)
#       and returns its  .content
#       Then call it twice - once with your Q5a object, once with your Q5b object.
#       The function itself does not change. That is the whole point of the wrapper.
#
# Ask both: "In one sentence, what is machine learning?"


# your code here

---

### ✅ What you practised

| Door | The one line | What comes back |
|---|---|---|
| **`openai`** | `client.chat.completions.create(model=..., messages=[...])` | `.choices[0].message.content` |
| **`openai` → Gemini** | same class, `base_url=".../v1beta/openai/"` | identical shape |
| **`litellm.completion`** | `completion(model="provider/model", messages=[...])` | identical shape again |
| **`litellm.embedding`** | `embedding(model="text-embedding-3-small", input=[...])` | `.data[i]["embedding"]` → 1536 numbers |
| **`sentence-transformers`** | `SentenceTransformer("all-MiniLM-L6-v2").encode([...])` | 384 numbers, free, local |
| **`langchain_openai`** | `ChatOpenAI(model=...).invoke("...")` | a message object → `.content` |
| **`langchain_google_genai`** | `ChatGoogleGenerativeAI(model=...).invoke("...")` | a message object → `.content` |

**The pattern worth keeping.** All seven calls do the same thing. What actually differs is *how
much you rewrite when you switch provider*: a whole new client (Q1), one string (Q2), or nothing
at all (Q5).

**Finished early?**

1. Add `dimensions=256` to the Q3 `embedding()` call. Do the Q3 rankings survive the shorter vector?
2. Set `temperature=1.5` on the Q5a `ChatOpenAI` and run it three times. Then do the same to your Q1a call. Same knob, same effect?
3. In Q2b, misspell the provider prefix on purpose — `"gemeni/..."`. Read the error. Would that message have told you what was wrong if you had not written the line yourself?
4. Break Q4 on purpose: `cosine(vectors[0], hf_vectors[0])` — a 1536-number vector against a 384-number one. Predict what happens *before* you run it.